# Imports

In [ ]:
from Fcts_FE import make_experiment
from Fcts_Base import get_stainings, find_zarr_dirs, save_adata

import os
import pandas as pd
import anndata as ad
from anndata import AnnData
from anndata.io import read_zarr

%load_ext autoreload
%autoreload 2

# User Input

In [ ]:
# Point to experiment folder. Has to include setup .xls file.
source = "YOUR PATH TO EXPERIMENT FOLDER"
analysis_dir = None
experiment_ID = "YOUR EXPERIMENT ID"

# Feature table(s) to load (str or list[str])
feature_table_names = "YOUR_FEATURE_TABLE_NAME"

# ROI table + label for downstream image/mask lookup
roi_table_name = "nuclei_ROI_table"
label_name = "nuclei"

file_ending = ".zarr"
result_file_name = "1_FeatureLoading"

# Load files and display experimental setup

In [ ]:
stainings = get_stainings(source)
folder = find_zarr_dirs(source, file_ending=file_ending)
experiment_setup, barcodes = make_experiment(source)

if analysis_dir is None:
    analysis_dir = source

# Load precomputed features from tables/<name>

In [ ]:
# Normalize feature_table_names input
if isinstance(feature_table_names, str):
    feature_table_names = [feature_table_names]
if len(feature_table_names) == 0:
    raise ValueError('feature_table_names is empty. Provide at least one table name.')

from ez_zarr import ome_zarr

def _load_plate(plate_path: str):
    return ome_zarr.import_plate(plate_path)

def _read_table_anndata(table_zarr_path: str) -> AnnData:
    return read_zarr(table_zarr_path)

def _stringify_dict_keys(obj):
    # H5AD writer struggles with non-string keys in nested dicts inside .uns
    if isinstance(obj, dict):
        return {str(k): _stringify_dict_keys(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_stringify_dict_keys(x) for x in obj]
    return obj

tables_all = []

for plate_folder in folder:
    plate_path = os.path.join(source, plate_folder)
    plate = _load_plate(plate_path)

    wells = plate.get_names()
    well_paths = plate.paths

    barcode_guess = None
    for bc in experiment_setup.keys():
        if bc in plate_folder:
            barcode_guess = bc
            break
    if barcode_guess is None:
        barcode_guess = list(experiment_setup.keys())[folder.index(plate_folder)]

    meta = experiment_setup[barcode_guess]

    for well, path_in_plate in zip(wells, well_paths):
        ad_list = []
        for tname in feature_table_names:
            table_path = os.path.join(plate_path, path_in_plate, 'tables', tname)
            if not os.path.exists(table_path):
                raise FileNotFoundError(f'Cannot find table: {table_path}')
            ad_t = _read_table_anndata(table_path)
            ad_list.append(ad_t)

        obs0 = ad_list[0].obs_names
        for j, ad_t in enumerate(ad_list[1:], start=1):
            if ad_t.n_obs != ad_list[0].n_obs:
                raise ValueError(f'n_obs mismatch in well {well} between table {feature_table_names[0]} and {feature_table_names[j]}')
            if not obs0.equals(ad_t.obs_names):
                raise ValueError(f'obs_names order mismatch in well {well} between table {feature_table_names[0]} and {feature_table_names[j]}')

        ad_list_pref = []
        for tname, ad_t in zip(feature_table_names, ad_list):
            ad_cp = ad_t.copy()
            ad_cp.var_names = [f'{tname}__{v}' for v in ad_cp.var_names]
            ad_list_pref.append(ad_cp)
        ad_well = ad.concat(ad_list_pref, axis=1, merge='same', join='outer')

        ad_well.obs = ad_well.obs.copy()
        ad_well.obs['Barcode'] = barcode_guess
        ad_well.obs['Well'] = well
        ad_well.obs['PATH'] = path_in_plate

        idx_in_well = pd.Series(range(ad_well.n_obs), index=ad_well.obs_names)
        ad_well.obs['Organoid_ID'] = barcode_guess + '-' + well + '-' + idx_in_well.astype(str).values

        exp_info = meta.get(well, [None, None, None, None])
        ad_well.obs['Medium'] = exp_info[0]
        ad_well.obs['ABs'] = exp_info[1]
        ad_well.obs['Cell_line'] = exp_info[2]
        ad_well.obs['Other'] = exp_info[3] if len(exp_info) > 3 else None
        ad_well.obs['Experiment_ID'] = experiment_ID

        tables_all.append(ad_well)

if len(tables_all) == 0:
    raise RuntimeError('No feature tables loaded. Check table names and OME-Zarr structure.')

ad_all = ad.concat(tables_all, axis=0, merge='same', join='outer', index_unique=None)
ad_all.obs = ad_all.obs.copy()
ad_all.obs_names = ad_all.obs['Organoid_ID'].astype(str)

# .uns metadata (stringify nested dict keys for safe h5ad writing)
ad_all.uns['stainings'] = _stringify_dict_keys(stainings)
ad_all.uns['experiment_setup'] = _stringify_dict_keys(experiment_setup)
ad_all.uns['folders'] = folder
ad_all.uns['source_dir'] = source
ad_all.uns['table_name'] = roi_table_name
ad_all.uns['label_name'] = label_name
ad_all.uns['feature_table_names'] = feature_table_names
ad_all.uns['experiment_ID'] = experiment_ID

# Save location (keep same convention as FeatureExtraction: 2_Tables)
ad_all.uns['table_dir'] = os.path.join(analysis_dir, '2_Tables')

print('Pooled AnnData created:')
print('  n_obs:', ad_all.n_obs)
print('  n_vars:', ad_all.n_vars)
print('  table_dir:', ad_all.uns['table_dir'])

# Save pooled AnnData

In [ ]:
save_adata(ad_all, result_file_name)